# 6th MODE Workshop Hackaton

## Part 1: Detector position optimization

Recommended Python packages:
- Python 3.14
- Numpy
- Pandas
- Matplotlib
- h5py
- PyTorch (can be installed from https://pytorch.org/get-started/locally/, depending on setup and CUDA version)

### Import necessary packages

In [1]:
%pip install numpy pandas matplotlib h5py

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd

# Enable CUDA allocator optimization
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import h5py

import torch
import torch.nn as nn
import torch.nn.functional as F

### Load functions

In [3]:
def load_to_memmap(file_list, filename='opensky_combined.dat', dtype=np.float32, N=-1):
    # Determine total dimensions
    shapes = []
    for f in file_list:
        with h5py.File(f, 'r') as h5:
            shape = h5['df/block0_values'].shape
            if shape[0] < shape[1]:
                shape = (shape[1], shape[0])
            shapes.append(shape)

    total_rows = sum(s[0] for s in shapes)
    n_features = shapes[0][1]

    print(f"Data to load: {file_list}")
    print(f"  Total number of rows: {total_rows}")
    print(f"  Number of features:   {n_features}")
    if N>0: print(f"   Will load {N} events")

    # Create memory-mapped file on disk
    if N>0 and total_rows>N:
        mmap_array = np.memmap(filename, dtype=dtype, mode='w+', shape=(N, n_features))
    else:
        mmap_array = np.memmap(filename, dtype=dtype, mode='w+', shape=(total_rows, n_features))

    current_idx = 0
    for f, shape in zip(file_list, shapes):
        with h5py.File(f, 'r') as h5:
            data = h5['df/block0_values'][:]
            if data.shape[0] < data.shape[1]:
                data = data.T
            rows = shape[0]
            if N>0 and current_idx+rows>N:
                mmap_array[current_idx:N] = data[0:N,:].astype(dtype, copy=False)
                current_idx = N
            else:
                mmap_array[current_idx:current_idx + rows] = data.astype(dtype, copy=False)
                current_idx += rows
                
    # Flush changes to disk
    mmap_array.flush()
    return mmap_array

In [4]:
def prepare_projected_tracks(
    data: np.ndarray,
    det_search_bbox: tuple[float, float, float, float],
    angle_bounds: float,
    margin: float = 200.0,
    detSizeZ: float = 40.0,
    Z: float = 800.0,
    xrange: tuple[float, float] = (-1500.,1500.),
    nbins: int = 50,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Pre-filters and pre-projects rays on CPU to avoid computing projection math
    inside the GPU autograd loop.
    Returns:
        track_pos: (N, 4) -> [x_top, y_top, x_bot, y_bot] (float32 on CPU)
        proj_idx:  (N, 4) -> [y0_c, x0_c, y1_c, x1_c] (int64 on CPU)
        interp_w:  (N, 4) -> [w00, w01, w10, w11] (float32 on CPU)
    """
    xmin, xmax, ymin, ymax = det_search_bbox
    dz = data[:, 5]
    theta = np.arctan(np.sqrt(data[:, 3] ** 2 + data[:, 4] ** 2) / -dz)

    # Prefilter the dataset
    mask = (
        (data[:, 0] >= xmin - margin)
        & (data[:, 0] <= xmax + margin)
        & (data[:, 1] >= ymin - margin)
        & (data[:, 1] <= ymax + margin)
        & (theta <= angle_bounds)
    )
    d = data[mask]

    # Precalculate geometric tracks
    # x_proj, y_proj: coordinates of muon at chosen Z value
    # x_top, y_top: coordinates of muon at Z = z_detector
    # x_bot, y_bot: coordinates of muon at Z = z_detector - detSizeZ
    dz = d[:, 5]
    l = (Z - d[:, 2]) / dz
    x_proj = d[:, 0] + l * d[:, 3]
    y_proj = d[:, 1] + l * d[:, 4]

    x_top, y_top = d[:, 0], d[:, 1]
    x_bot = d[:, 0] - (detSizeZ / dz) * d[:, 3]
    y_bot = d[:, 1] - (detSizeZ / dz) * d[:, 4]

    track_pos = torch.from_numpy(
        np.column_stack([x_top, y_top, x_bot, y_bot]).astype(np.float32)
    )

    # Precalculate static 2D histogram splatting weights and indices
    x_norm = np.clip((x_proj - (xrange[0])) / (xrange[1]-xrange[0]) * float(nbins-1), 0, nbins-1)
    y_norm = np.clip((y_proj - (xrange[0])) / (xrange[1]-xrange[0]) * float(nbins-1), 0, nbins-1)

    x0 = np.floor(x_norm).astype(np.int64)
    y0 = np.floor(y_norm).astype(np.int64)
    x1 = np.clip(x0 + 1, 0, nbins-1)
    y1 = np.clip(y0 + 1, 0, nbins-1)

    wx1 = (x_norm - x0).astype(np.float32)
    wx0 = (1.0 - wx1).astype(np.float32)
    wy1 = (y_norm - y0).astype(np.float32)
    wy0 = (1.0 - wy1).astype(np.float32)

    w00 = wx0 * wy0
    w01 = wx0 * wy1
    w10 = wx1 * wy0
    w11 = wx1 * wy1

    proj_idx = torch.from_numpy(np.column_stack([y0, x0, y1, x1]))
    interp_w = torch.from_numpy(np.column_stack([w00, w01, w10, w11]))

    return track_pos, proj_idx, interp_w

In [5]:
class DifferentiableHistogram2D(nn.Module):
    def __init__(self, bins: tuple[int, int] = (50, 50), x_range: tuple[float, float] = (-1500.0, 1500.0), y_range: tuple[float, float] = (-1500.0, 1500.0)):
        super().__init__()
        self.H_bins, self.W_bins = bins
        self.x_min, self.x_max = x_range
        self.y_min, self.y_max = y_range

    def forward(self, x_coords: torch.Tensor, y_coords: torch.Tensor, weights: torch.Tensor = None) -> torch.Tensor:
        """Args:
        x_coords: 1D Tensor of shape (N,)
        y_coords: 1D Tensor of shape (N,)
        weights: Optional 1D Tensor of shape (N,)
        Returns:
            2D continuous histogram of shape (H_bins, W_bins)
        """
        if weights is None:
            weights = torch.ones_like(x_coords)

        # 1. Normalize coordinates to continuous grid index coordinates [0, bins-1]
        x_norm = (x_coords - self.x_min) / (self.x_max - self.x_min) * (self.W_bins - 1)
        y_norm = (y_coords - self.y_min) / (self.y_max - self.y_min) * (self.H_bins - 1)

        # 2. Get bounding 4 neighbor cell indices
        x0 = torch.floor(x_norm).long()
        y0 = torch.floor(y_norm).long()
        x1 = x0 + 1
        y1 = y0 + 1

        # Clip indices to grid boundaries to avoid indexing errors
        x0_c = torch.clamp(x0, 0, self.W_bins - 1)
        x1_c = torch.clamp(x1, 0, self.W_bins - 1)
        y0_c = torch.clamp(y0, 0, self.H_bins - 1)
        y1_c = torch.clamp(y1, 0, self.H_bins - 1)

        # 3. Bilinear interpolation weights (differentiable w.r.t coordinates)
        w_x1 = x_norm - x0.float()
        w_x0 = 1.0 - w_x1
        w_y1 = y_norm - y0.float()
        w_y0 = 1.0 - w_y1

        w00 = weights * w_x0 * w_y0
        w01 = weights * w_x0 * w_y1
        w10 = weights * w_x1 * w_y0
        w11 = weights * w_x1 * w_y1

        # 4. Scatter add into 2D grid
        hist = torch.zeros(
            (self.H_bins, self.W_bins),
            dtype=x_coords.dtype,
            device=x_coords.device,
        )

        hist.index_put_((y0_c, x0_c), w00, accumulate=True)
        hist.index_put_((y1_c, x0_c), w01, accumulate=True)
        hist.index_put_((y0_c, x1_c), w10, accumulate=True)
        hist.index_put_((y1_c, x1_c), w11, accumulate=True)

        return hist

In [6]:
class ChamberBlindDetectorLoss(nn.Module):

    def __init__(self, kernel_size: int = 15, sigma: float = 3.0, temperature: float = 0.1):
        super().__init__()
        self.temperature = temperature

        # Create a 2D Gaussian smoothing kernel
        ax = torch.arange(-kernel_size // 2 + 1.0, kernel_size // 2 + 1.0)
        xx, yy = torch.meshgrid(ax, ax, indexing="ij")
        kernel = torch.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
        kernel = kernel / kernel.sum()

        # Shape (out_channels, in_channels, H, W) for conv2d
        self.register_buffer("kernel", kernel.view(1, 1, kernel_size, kernel_size))

    def forward(self, ratio_map: torch.Tensor, sky_counts: torch.Tensor = None, crop_margin: int = 8):
        """Args:
        ratio_map: Tensor of shape (1, 1, H, W) or (H, W)
        valid_mask: Optional boolean mask (1 for valid sky bins, 0 for edge
        voids)
        """
        if ratio_map.ndim == 2:
            ratio_map = ratio_map.unsqueeze(0).unsqueeze(0)

        # Spatially smooth to aggregate chamber signal and suppress single-pixel noise
        smoothed = F.conv2d(ratio_map, self.kernel, padding="same")

        # Build spatial validity mask to cut out boundary noise
        H, W = ratio_map.shape[-2], ratio_map.shape[-1]
        spatial_mask = torch.zeros((H, W), dtype=torch.bool, device=ratio_map.device)
        spatial_mask[crop_margin : H - crop_margin, crop_margin : W - crop_margin] = True
        
        if sky_counts is not None:
            # Exclude bins with poor statistics
            stat_mask = (sky_counts.squeeze() > 5.0) & spatial_mask
            flat_smoothed = smoothed.squeeze()[stat_mask]
        else:
            flat_smoothed = smoothed.squeeze()[spatial_mask]

        # Background statistics
        mu_bg = torch.mean(flat_smoothed)
        sigma_bg = torch.std(flat_smoothed) + 1e-6

        # SoftMax-based peak approximation (smooth, differentiable max)
        # LogSumExp gives a smooth approximation of max(smoothed)
        peak_val = (self.temperature * torch.logsumexp(flat_smoothed / self.temperature, dim=0) - self.temperature * torch.log(torch.tensor(flat_smoothed.numel())))

        # Significance Z-score
        z_score = (peak_val - mu_bg) / sigma_bg

        # Return negative Z-score for minimization
        print(f"Peak value = {peak_val:.3f}")
        print(f"Background mean value = {mu_bg:.3f}")
        print(f"Background sigma value = {sigma_bg:.3f}")
        print(f"Z score = {z_score:.3f}")
        return -z_score

In [7]:
class ChamberDoGLoss(nn.Module):

    def __init__(self, kernel_size: int = 20, sigma1: float = 2.0, sigma2: float = 5.0, temperature: float = 0.05):
        super().__init__()
        self.temperature = temperature

        # Create coordinate grid
        ax = torch.arange(-kernel_size // 2 + 1.0, kernel_size // 2 + 1.0)
        xx, yy = torch.meshgrid(ax, ax, indexing="ij")
        r2 = xx**2 + yy**2

        # Compute both Gaussian components
        g1 = torch.exp(-r2 / (2.0 * sigma1**2))
        g1 = g1 / g1.sum()

        g2 = torch.exp(-r2 / (2.0 * sigma2**2))
        g2 = g2 / g2.sum()

        # DoG kernel (zero-mean bandpass filter)
        dog_kernel = g1 - g2
        self.register_buffer("kernel", dog_kernel.view(1, 1, kernel_size, kernel_size))

    def forward(self, ratio_map: torch.Tensor, sky_counts: torch.Tensor = None, crop_margin: int = 8):
        """Args:
        ratio_map: Tensor (H, W) or (1, 1, H, W)
        sky_counts: Tensor (H, W) of OpenSky counts (used for statistical
        masking)
        crop_margin: Number of boundary bins to strip from the outer edges
        """
        if ratio_map.ndim == 2:
            ratio_map = ratio_map.unsqueeze(0).unsqueeze(0)

        # 1. Bandpass filter to suppress edge gradients and single-pixel spikes
        dog_response = F.conv2d(ratio_map, self.kernel, padding="same")

        # 2. Build spatial validity mask to cut out boundary noise
        H, W = ratio_map.shape[-2], ratio_map.shape[-1]
        spatial_mask = torch.zeros((H, W), dtype=torch.bool, device=ratio_map.device)
        spatial_mask[crop_margin : H - crop_margin, crop_margin : W - crop_margin] = True

        if sky_counts is not None:
            # Exclude bins with poor statistics
            stat_mask = (sky_counts.squeeze() > 5.0) & spatial_mask
            flat_dog = dog_response.squeeze()[stat_mask]
        else:
            flat_dog = dog_response.squeeze()[spatial_mask]

        # 3. Standardize response over the valid interior
        mu = torch.mean(flat_dog)
        sigma = torch.std(flat_dog) + 1e-6

        # 4. Smooth, differentiable peak extraction via LogSumExp
        # SoftMax temperature controls peak sharpness
        peak = self.temperature * torch.logsumexp((flat_dog - mu) / (sigma * self.temperature), dim=0)

        peak_val = (self.temperature * torch.logsumexp(flat_dog / self.temperature, dim=0) - self.temperature * torch.log(torch.tensor(flat_dog.numel())))

        # Significance Z-score
        z_score = (peak_val - mu) / sigma

        # Minimize negative peak response
        return -z_score

In [8]:
def soft_box_2d(x: torch.Tensor, y: torch.Tensor, center: torch.Tensor, size_xy: float, temperature: float = 1.0) -> torch.Tensor:
    """Computes a smooth differentiable acceptance weight in [0, 1]
    for points (x, y) falling inside the detector box centered at `center`.
    """
    half_w = size_xy / 2.0
    x_min, x_max = center[0] - half_w, center[0] + half_w
    y_min, y_max = center[1] - half_w, center[1] + half_w

    # Smooth step function using sigmoids
    in_x = torch.sigmoid((x - x_min) / temperature) * torch.sigmoid((x_max - x) / temperature)
    in_y = torch.sigmoid((y - y_min) / temperature) * torch.sigmoid((y_max - y) / temperature)
    return in_x * in_y

In [9]:
def project_and_accumulate_streaming(track_pos: torch.Tensor, proj_idx: torch.Tensor, interp_w: torch.Tensor, detPos: torch.Tensor,
                                     detSize: float = 100.0, temp: float = 2.0, chunk_size: int = 150_000,) -> torch.Tensor:
    """Streams data from CPU RAM to GPU chunk by chunk."""
    device = detPos.device
    H_bins, W_bins = 50, 50
    hist_acc = torch.zeros((H_bins, W_bins), dtype=torch.float32, device=device)

    total_rows = track_pos.shape[0]

    for i in range(0, total_rows, chunk_size):
        # 1. Asynchronously send small chunk to GPU
        b_pos = track_pos[i : i + chunk_size].to(device, non_blocking=True)
        b_idx = proj_idx[i : i + chunk_size].to(device, non_blocking=True)
        b_iw = interp_w[i : i + chunk_size].to(device, non_blocking=True)

        x_top, y_top = b_pos[:, 0], b_pos[:, 1]
        x_bot, y_bot = b_pos[:, 2], b_pos[:, 3]

        # 2. Differentiable acceptance weights for all 3 detectors
        weights = torch.zeros(b_pos.shape[0], dtype=torch.float32, device=device)
        for k in range(detPos.shape[0]):
            pos = detPos[k]
            w_top = soft_box_2d(x_top, y_top, pos, detSize, temperature=temp)
            w_bot = soft_box_2d(x_bot, y_bot, pos, detSize, temperature=temp)
            weights = weights + (w_top * w_bot)

        # 3. Accumulate soft-binned histogram
        w00 = weights * b_iw[:, 0]
        w01 = weights * b_iw[:, 1]
        w10 = weights * b_iw[:, 2]
        w11 = weights * b_iw[:, 3]

        y0, x0 = b_idx[:, 0], b_idx[:, 1]
        y1, x1 = b_idx[:, 2], b_idx[:, 3]

        hist_batch = torch.zeros((H_bins, W_bins), dtype=torch.float32, device=device)
        hist_batch.index_put_((y0, x0), w00, accumulate=True)
        hist_batch.index_put_((y1, x0), w01, accumulate=True)
        hist_batch.index_put_((y0, x1), w10, accumulate=True)
        hist_batch.index_put_((y1, x1), w11, accumulate=True)

        hist_acc = hist_acc + hist_batch

    return hist_acc

In [10]:
def project_non_overlapping_(
    pos: torch.Tensor,
    det_size: float = 100.0,
    boundaries: list[float] = [-400.0, 400.0, -600.0, 600.0],
    max_iters: int = 20,
):
    """
    Hard in-place geometric projection:
    1. Projects overlapping detector pairs apart along the minimum-penetration axis.
    2. Clamps all detectors strictly within domain boundaries.
    """
    half_w = det_size / 2.0
    x_min, x_max = boundaries[0] + half_w, boundaries[1] - half_w
    y_min, y_max = boundaries[2] + half_w, boundaries[3] - half_w
    K = pos.shape[0]

    for _ in range(max_iters):
        collision_detected = False

        for i in range(K):
            for j in range(i + 1, K):
                dx = pos[j, 0] - pos[i, 0]
                dy = pos[j, 1] - pos[i, 1]

                pen_x = det_size - torch.abs(dx)
                pen_y = det_size - torch.abs(dy)

                # Overlap exists only if penetrating along both axes
                if pen_x > 0 and pen_y > 0:
                    collision_detected = True

                    # Separate along the shallowest penetration axis
                    if pen_x < pen_y:
                        direction = 1.0 if dx >= 0 else -1.0
                        pos[i, 0] -= (pen_x / 2.0) * direction
                        pos[j, 0] += (pen_x / 2.0) * direction
                    else:
                        direction = 1.0 if dy >= 0 else -1.0
                        pos[i, 1] -= (pen_y / 2.0) * direction
                        pos[j, 1] += (pen_y / 2.0) * direction

        # Hard boundary clamping
        pos[:, 0].clamp_(min=x_min, max=x_max)
        pos[:, 1].clamp_(min=y_min, max=y_max)

        if not collision_detected:
            break

### Data loading

In [11]:
openskyFiles = ['data/opensky.h5']
tunnelFiles = ['data/tunnel.h5']

openskyfull = load_to_memmap(openskyFiles, 'opensky.dat')
tunnelfull  = load_to_memmap(tunnelFiles, 'tunnel.dat')

print(">>>> Pre-calculating projected geometry on CPU...")
tunnel_bounds = [-400.0, 400.0, -1000.0, 1000.0]
angle_bounds = np.pi / 2.

opensky_pos, opensky_idx, opensky_iw = prepare_projected_tracks(openskyfull, tunnel_bounds, angle_bounds, Z=800.0)
tunnel_pos, tunnel_idx, tunnel_iw = prepare_projected_tracks(tunnelfull, tunnel_bounds, angle_bounds, Z=800.0)

# Pin memory for fast CPU -> GPU async streaming transfers
tunnel_pos, tunnel_idx, tunnel_iw = (
    tunnel_pos.pin_memory(),
    tunnel_idx.pin_memory(),
    tunnel_iw.pin_memory(),
)
opensky_pos, opensky_idx, opensky_iw = (
    opensky_pos.pin_memory(),
    opensky_idx.pin_memory(),
    opensky_iw.pin_memory(),
)

Data to load: ['data/opensky.h5']
  Total number of rows: 80350277
  Number of features:   6
Data to load: ['data/tunnel.h5']
  Total number of rows: 34465886
  Number of features:   6
>>>> Pre-calculating projected geometry on CPU...


### Optimization loop

In [12]:
# Create plots directory if it does not exist
os.makedirs("plots_jupyter", exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

LEARNING_RATE = 40.0
XRANGE = (-1500.0, 1500.0)
initDetectorPos = [[0.0, -100.0], [0.0, 0.0], [0.0, 200.0]]

detector_pos = torch.nn.Parameter(
    torch.tensor(
        initDetectorPos,
        dtype=torch.float32,
        device=device,
        requires_grad=True,
    )
)
optimizer = torch.optim.Adam([detector_pos], lr=LEARNING_RATE)
blind_loss_peak = ChamberBlindDetectorLoss(kernel_size=8, sigma=2.0).to(device)
blind_loss_DoG = ChamberDoGLoss().to(device)

print(">>>> Starting optimization loop...")
for epoch in range(50):
    optimizer.zero_grad(set_to_none=True)

    H_tunnel = project_and_accumulate_streaming(
        tunnel_pos,
        tunnel_idx,
        tunnel_iw,
        detector_pos,
        detSize=100.0,
        temp=2.0,
        chunk_size=150_000,
    )
    H_sky = project_and_accumulate_streaming(
        opensky_pos,
        opensky_idx,
        opensky_iw,
        detector_pos,
        detSize=100.0,
        temp=2.0,
        chunk_size=150_000,
    )

    ratio_map = H_tunnel / (H_sky + 1e-3)

    loss_peak = blind_loss_peak(ratio_map, sky_counts = H_sky)
    loss_DoG = blind_loss_DoG(ratio_map, sky_counts = H_sky)
    loss_DoG.backward()
    optimizer.step()

    # Enforce hard constraints (non-overlap + domain boundaries)
    detPos_boundaries = [-400.0, 400.0, -600.0, 600.0]
    with torch.no_grad():
        project_non_overlapping_(detector_pos, det_size=100.0, boundaries=detPos_boundaries)

    print(
        f"Epoch {epoch+1:02d} | Loss: {loss_DoG.item():.4f} | "
        f"Det 0: {detector_pos[0].detach().cpu().numpy().round(1)} | "
        f"Det 1: {detector_pos[1].detach().cpu().numpy().round(1)} | "
        f"Det 2: {detector_pos[2].detach().cpu().numpy().round(1)}"
    )

    with torch.no_grad():
        # Plot epoch ratio hist
        fig,ax = plt.subplots(2, 2, figsize = (18, 12))
        xedges = np.linspace(XRANGE[0], XRANGE[1], 51)
        yedges = np.linspace(XRANGE[0], XRANGE[1], 51)
        pc = ax[0][0].pcolorfast(xedges, yedges, ratio_map.detach().cpu().numpy().T)
        fig.colorbar(pc, ax=ax[0][0])
        smoothed = F.conv2d(ratio_map.unsqueeze(0).unsqueeze(0), blind_loss_DoG.kernel, padding="same")
        H, W = 50, 50
        crop_margin=8
        spatial_mask = torch.zeros((H, W), dtype=torch.bool, device=ratio_map.device)
        spatial_mask[crop_margin : H - crop_margin, crop_margin : W - crop_margin] = True
        print(smoothed[0][0].shape,(smoothed[0][0]*spatial_mask).shape)
        pc1 = ax[0][1].pcolorfast(xedges, yedges, np.array(smoothed[0][0].detach().cpu().numpy()).T)
        fig.colorbar(pc1, ax=ax[0][1])
        pc2 = ax[1][0].pcolorfast(xedges, yedges, np.array((smoothed[0][0]*spatial_mask).detach().cpu().numpy()).T)
        fig.colorbar(pc2, ax=ax[1][0])
        detPos1 = detector_pos[0].detach().cpu().numpy()
        detPos2 = detector_pos[1].detach().cpu().numpy()
        detPos3 = detector_pos[2].detach().cpu().numpy()
        det1 = patches.Rectangle((detPos1[0]-50,detPos1[1]-50), 100, 100, edgecolor='red', facecolor='none', linewidth=2)
        det2 = patches.Rectangle((detPos2[0]-50,detPos2[1]-50), 100, 100, edgecolor='red', facecolor='none', linewidth=2)
        det3 = patches.Rectangle((detPos3[0]-50,detPos3[1]-50), 100, 100, edgecolor='red', facecolor='none', linewidth=2)
        ax[1][1].add_patch(det1)
        ax[1][1].add_patch(det2)
        ax[1][1].add_patch(det3)
        ax[1][1].set_ylim(-500,500)
        ax[1][1].set_xlim(-500,500)
        plt.savefig(f'plots_jupyter/epoch_{epoch}.png')
        plt.close(fig)

Device: cuda
>>>> Starting optimization loop...


/tmp/ipykernel_1040368/1557376126.py:26: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1101.)
  smoothed = F.conv2d(ratio_map, self.kernel, padding="same")


Peak value = 0.461
Background mean value = 0.456
Background sigma value = 0.032
Z score = 0.172
Epoch 01 | Loss: -0.1588 | Det 0: [ 50. -60.] | Det 1: [-50. -40.] | Det 2: [ 40. 160.]
torch.Size([50, 50]) torch.Size([50, 50])
Peak value = 0.460
Background mean value = 0.455
Background sigma value = 0.032
Z score = 0.180
Epoch 02 | Loss: -0.1714 | Det 0: [ 41.  -38.3] | Det 1: [-59.  -30.5] | Det 2: [ 73.8 125.9]
torch.Size([50, 50]) torch.Size([50, 50])
Peak value = 0.459
Background mean value = 0.453
Background sigma value = 0.032
Z score = 0.180
Epoch 03 | Loss: -0.1805 | Det 0: [ 36.9 -24.5] | Det 1: [-63.1 -48.3] | Det 2: [62.4 94.8]
torch.Size([50, 50]) torch.Size([50, 50])
Peak value = 0.460
Background mean value = 0.453
Background sigma value = 0.034
Z score = 0.190
Epoch 04 | Loss: -0.2001 | Det 0: [ 32.2 -19.2] | Det 1: [-67.8 -42.3] | Det 2: [77.  80.8]
torch.Size([50, 50]) torch.Size([50, 50])
Peak value = 0.459
Background mean value = 0.453
Background sigma value = 0.034
Z 